# 08 – Kaggle-Daten manuell herunterladen und prüfen

**Ziel:** Einen Datensatz auswählen, im Browser herunterladen, richtig ablegen, kurz prüfen und anschließend mit pandas ansehen. Kein Kaggle-Python-Paket und keine API-Einrichtung nötig. Kaggle kann auch für den Browser-Download eine Anmeldung verlangen; bei Wettbewerben müssen die Regeln akzeptiert werden.

## 1. Downloadseite auswählen

| Datensatz | URL zum Download und zur Dokumentation | Zielordner relativ zum Kursordner |
|---|---|---|
| Titanic | [Titanic – Data](https://www.kaggle.com/competitions/titanic/data) | `set3-datenanalyse-und-visualisierung/daten/kaggle/titanic/` |
| House Prices | [House Prices – Data](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data) | `set3-datenanalyse-und-visualisierung/daten/kaggle/house-prices/` |
| Credit Card Fraud | [Credit Card Fraud – Data Card](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) | `set3-datenanalyse-und-visualisierung/daten/kaggle/credit-card/` |

## 2. Herunterladen und ablegen

1. Öffne den Link für deinen Datensatz. Übung 05 verwendet einen anderen Datensatz und enthält dafür eine eigene Downloadanleitung.
2. Melde dich auf Kaggle an, falls verlangt. Bei Titanic/House Prices gegebenenfalls dem Wettbewerb beitreten und die Regeln lesen und akzeptieren.
3. Auf der Data-Seite **Download All** bzw. **Download** wählen und die ZIP-Datei lokal speichern.
4. ZIP entpacken (unter Windows: Rechtsklick → Alle extrahieren). Den oben angegebenen Zielordner bei Bedarf erstellen und die entpackten Dateien **direkt** dort ablegen.
5. Darauf achten, dass kein zusätzlicher Unterordner wie `archive/` dazwischenliegt. Eine ZIP im Zielordner allein genügt nicht. Dateien verschiedener Datensätze getrennt halten, weil sie häufig gleich heißen.

Erwartete Dateien:

```text
set3-datenanalyse-und-visualisierung/daten/kaggle/
├── titanic/
│   ├── train.csv
│   ├── test.csv
│   └── gender_submission.csv
├── house-prices/
│   ├── train.csv
│   ├── test.csv
│   ├── sample_submission.csv
│   └── data_description.txt
└── credit-card/
    └── creditcard.csv
```

Du brauchst nur den gewählten Datensatz. Notiere Quelle und Abrufdatum; beachte die Nutzungsbedingungen auf der jeweiligen Seite.

In [ ]:
from pathlib import Path
import pandas as pd

auswahl = 'titanic'  # alternativ: 'house-prices' oder 'credit-card'
katalog = {
    'titanic': {'train.csv': ['PassengerId', 'Survived'],
                'test.csv': ['PassengerId'], 'gender_submission.csv': ['PassengerId', 'Survived']},
    'house-prices': {'train.csv': ['Id', 'SalePrice'], 'test.csv': ['Id'],
                     'sample_submission.csv': ['Id', 'SalePrice'], 'data_description.txt': None},
    'credit-card': {'creditcard.csv': ['Time', 'Amount', 'Class']},
}
wurzel = next((p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / 'set3-datenanalyse-und-visualisierung').is_dir()), None)
if wurzel is None:
    raise FileNotFoundError('Bitte das Notebook innerhalb des Kursordners öffnen.')
ordner = wurzel / 'set3-datenanalyse-und-visualisierung/daten/kaggle' / auswahl
print('Entpackte Dateien direkt hier ablegen:', ordner)

## 3. Dateien kurz prüfen

Nach dem Entpacken diese Zelle ausführen. Sie prüft Dateien, Lesbarkeit und einige erwartete Spalten. Pro CSV werden höchstens drei Zeilen gelesen; es werden keine Daten heruntergeladen oder verändert. Fehlende oder falsche Dateien werden mit einem Hinweis gemeldet.

Eine erfolgreiche Dateiprüfung bestätigt nur die grundlegende Ablage und Struktur, nicht Vollständigkeit oder Datenqualität.

In [ ]:
def dateien_pruefen(ordner, erwartet):
    fehler = []
    for name, spalten in erwartet.items():
        datei = ordner / name
        try:
            if not datei.is_file():
                raise FileNotFoundError('fehlt – ZIP entpacken und Datei direkt im Zielordner ablegen')
            if spalten is None:
                inhalt = datei.read_text(encoding='utf-8-sig')
                if not inhalt.strip():
                    raise ValueError('Textdatei ist leer')
            else:
                vorschau = pd.read_csv(datei, nrows=3)
                if vorschau.empty:
                    raise ValueError('CSV enthält keine Datenzeilen')
                fehlend = set(spalten) - set(vorschau.columns)
                if fehlend:
                    raise ValueError(f'Erwartete Spalten fehlen: {sorted(fehlend)}')
            print(f'OK: {name} ist vorhanden und lesbar.')
        except (OSError, UnicodeError, ValueError, pd.errors.ParserError) as exc:
            fehler.append(name)
            print(f'PRÜFEN: {name}: {exc}')
    if fehler:
        print('Zielordner:', ordner)
        print('Dateien korrigieren und diese Zelle erneut ausführen.')
    else:
        print('Dateien erfolgreich geprüft. Jetzt die Daten mit pandas ansehen.')
    return not fehler

alles_ok = dateien_pruefen(ordner, katalog[auswahl])

## 4. Ein kurzer Blick mit pandas

Nach erfolgreicher Dateiprüfung lesen wir die ausgewählte CSV vollständig ein. Bei Titanic und House Prices verwenden wir `train.csv`, bei Credit Card Fraud `creditcard.csv`.

`head()` zeigt die ersten fünf Zeilen, `shape` den Umfang, `info()` Spalten und Datentypen. Anschließend sehen wir nach fehlenden Werten. Das ist nur eine erste Orientierung; die ersten fünf Zeilen sind keine repräsentative Stichprobe.

**Kurz besprechen:** Was beschreibt eine Zeile? Welche Spalte ist das Ziel? Welche Spalten sind IDs oder Kategorien? Bei House Prices die Bedeutung von `NA` in der Datenbeschreibung prüfen: pandas kann damit auch „keine Garage“ oder „kein Keller“ als fehlenden Wert einlesen.

In [ ]:
dateiname = 'creditcard.csv' if auswahl == 'credit-card' else 'train.csv'
if alles_ok:
    daten = pd.read_csv(ordner / dateiname)
    print(f'{auswahl}: {daten.shape[0]:,} Zeilen, {daten.shape[1]} Spalten')
    display(daten.head())
    daten.info()
    fehlende_werte = daten.isna().sum().sort_values(ascending=False)
    print('Spalten mit fehlenden Werten (höchstens 10):')
    display(fehlende_werte[fehlende_werte > 0].head(10))
else:
    print('Zuerst die Hinweise aus der Dateiprüfung beheben, dann erneut ausführen.')

## 5. Mit der Dokumentation weiterarbeiten

Lies die Data-Seite und bei House Prices zusätzlich `data_description.txt`. Notiere Beobachtungseinheit, Zielvariable, Spaltenbedeutungen/Einheiten und die Bedeutung fehlender Werte. Nicht dokumentierte Angaben bleiben unbekannt.

Bei Titanic und House Prices enthält `train.csv` Zielwerte; `test.csv` enthält keine veröffentlichten Zielwerte. Die Submission-Dateien sind Formatbeispiele, keine wahren Testlabels.

Weiter mit `uebungen/05_uebung_datensatz_dokumentation.ipynb`: Den neuen Datensatz **Student Alcohol Consumption / Student Performance** manuell herunterladen und Fragen anhand seiner Dokumentation beantworten. Link und Zielordner stehen direkt im Aufgabenblatt. In dieser Übung liegt der Schwerpunkt auf der Dokumentation; sie verlangt keine EDA und kein Modelltraining.

**Was ist `gender_submission.csv` bei Titanic?** Die Datei zeigt das Format für eine Einreichung bei Kaggle: `PassengerId` identifiziert eine Person aus `test.csv`, `Survived` enthält die Vorhersage (1 = überlebt, 0 = nicht überlebt). Das Beispiel nutzt die einfache Regel „alle Frauen überleben, alle Männer nicht“. Das sind **Beispielvorhersagen, keine tatsächlichen Ergebnisse**. Deshalb die Datei nicht als wahre Testlabels verwenden. Für unseren ersten pandas-Blick verwenden wir `train.csv`. [Quelle: Kaggle-Datenbeschreibung](https://www.kaggle.com/competitions/titanic/data).
